Orders raw file

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

order_data = [
    Row(order_id=20001, customer_id=1001, order_date="01-03-2025", status="Delivered", created_date="01-03-2025", modified_date="01-03-2025"),
    Row(order_id=20002, customer_id=1002, order_date="01-03-2025", status="Delivered", created_date="01-03-2025", modified_date="04-03-2025"),
    Row(order_id=20003, customer_id=1003, order_date="02-03-2025", status="Delivered", created_date="02-03-2025", modified_date="07-03-2025"),
    Row(order_id=20004, customer_id=1004, order_date="02-03-2025", status="Cancelled", created_date="02-03-2025", modified_date="10-03-2025"),
    Row(order_id=20005, customer_id=1005, order_date="03-03-2025", status="Delivered", created_date="03-03-2025", modified_date="13-03-2025"),
    Row(order_id=20006, customer_id=1006, order_date="03-03-2025", status="Shipped", created_date="03-03-2025", modified_date="16-03-2025"),
    Row(order_id=20007, customer_id=1007, order_date="04-03-2025", status="Delivered", created_date="04-03-2025", modified_date="19-03-2025"),
    Row(order_id=20008, customer_id=1008, order_date="04-03-2025", status="Pending", created_date="04-03-2025", modified_date="22-03-2025"),
    Row(order_id=20009, customer_id=1009, order_date="05-03-2025", status="Delivered", created_date="05-03-2025", modified_date="25-03-2025"),
    Row(order_id=20010, customer_id=1010, order_date="05-03-2025", status="Delivered", created_date="05-03-2025", modified_date="28-03-2025")
]

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("order_date", StringType(), False),
    StructField("status", StringType(), False),
    StructField("created_date", StringType(), False),
    StructField("modified_date", StringType(), False)
])

orders_raw = spark.createDataFrame(order_data, order_schema)

orders_raw = orders_raw \
    .withColumn("order_date", to_timestamp("order_date", "dd-MM-yyyy")) \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

display(orders_raw)

order_id,customer_id,order_date,status,created_date,modified_date
20001,1001,2025-03-01T00:00:00.000Z,Delivered,2025-03-01T00:00:00.000Z,2025-03-01T00:00:00.000Z
20002,1002,2025-03-01T00:00:00.000Z,Delivered,2025-03-01T00:00:00.000Z,2025-03-04T00:00:00.000Z
20003,1003,2025-03-02T00:00:00.000Z,Delivered,2025-03-02T00:00:00.000Z,2025-03-07T00:00:00.000Z
20004,1004,2025-03-02T00:00:00.000Z,Cancelled,2025-03-02T00:00:00.000Z,2025-03-10T00:00:00.000Z
20005,1005,2025-03-03T00:00:00.000Z,Delivered,2025-03-03T00:00:00.000Z,2025-03-13T00:00:00.000Z
20006,1006,2025-03-03T00:00:00.000Z,Shipped,2025-03-03T00:00:00.000Z,2025-03-16T00:00:00.000Z
20007,1007,2025-03-04T00:00:00.000Z,Delivered,2025-03-04T00:00:00.000Z,2025-03-19T00:00:00.000Z
20008,1008,2025-03-04T00:00:00.000Z,Pending,2025-03-04T00:00:00.000Z,2025-03-22T00:00:00.000Z
20009,1009,2025-03-05T00:00:00.000Z,Delivered,2025-03-05T00:00:00.000Z,2025-03-25T00:00:00.000Z
20010,1010,2025-03-05T00:00:00.000Z,Delivered,2025-03-05T00:00:00.000Z,2025-03-28T00:00:00.000Z


In [0]:
# spark.sql("DROP TABLE IF EXISTS catalog_project1.source1.orders_raw")
orders_raw.write.mode("append")\
                  .format("delta")\
                  .option("mergeSchema", "true")\
                  .saveAsTable("catalog_project1.source1.orders_raw")

In [0]:
spark.sql("""
SELECT * 
FROM catalog_project1.source1.orders_raw 
WHERE to_date(modified_date, 'dd-MM-yyyy') < to_date(created_date, 'dd-MM-yyyy')
""").display()

order_id,customer_id,order_date,status,created_date,modified_date
